In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession

from utils.spark import create_spark, read_postgres
from utils.connector import postgres

def query(sql, params=None):
    with postgres() as pg:
        return pg.query(sql, params)

In [ ]:
sql = """WITH filtered AS (
    SELECT
        a.activity_id,
        a.assay_id,
        a.molregno,
        cs.canonical_smiles,
        a.standard_value,
        a.standard_units,
        a.standard_type,
        a.relation,
        a.pchembl_value,
        a.data_validity_comment,
        a.activity_comment,
        a.src_id,

        -- pIC50 computation (IC50 in nM)
        -LOG(10, a.standard_value * 1e-9) AS pIC50

    FROM public.activities a
    JOIN public.compound_structures cs
      ON a.molregno = cs.molregno

    WHERE
        a.standard_type = 'IC50'
        AND a.standard_units = 'nM'
        AND a.relation = '='
        AND a.standard_value IS NOT NULL
        AND a.standard_value > 0
        AND a.data_validity_comment IS NULL
)

SELECT
    canonical_smiles,
    COUNT(*)               AS n_assays,
    AVG(pIC50)              AS pIC50_mean,
    PERCENTILE_CONT(0.5) 
        WITHIN GROUP (ORDER BY pIC50) AS pIC50_median,
    MIN(pIC50)              AS pIC50_min,
    MAX(pIC50)              AS pIC50_max
FROM filtered
GROUP BY canonical_smiles;
"""

data = query(sql)